# Categorizing Pull Requests with NLP

This notebook explores pull request titles and bodies, builds an initial taxonomy, and assigns keyword-based PR categories for downstream analysis.

## 1. Imports and reproducibility

Load the libraries used for data processing, NLP modeling, visualization, and reproducible analysis.

In [2]:
from __future__ import annotations

import gc
import html
import json
import math
import re
import warnings
from collections import Counter
from pathlib import Path
from typing import Iterable

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse

from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    classification_report,
    f1_score,
    hamming_loss,
    silhouette_score,
)
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MultiLabelBinarizer
from wordcloud import WordCloud

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 180)

C:\Users\sydne\AppData\Local\Temp\ipykernel_34372\2222741009.py:17: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.5.4)
  from scipy import sparse


## 2. Load the dataset

Read the cleaned pull request title/body dataset and inspect its basic shape before analysis.

In [2]:
df_title_body = pd.read_csv("../data/cleaned_public.csv")

print(df_title_body.shape)
df_title_body.head()

print("Rows:", df_title_body.shape[0])
print("Columns:", df_title_body.shape[1])
print("Shape:", df_title_body.shape)

(1338401, 37)
Rows: 1338401
Columns: 37
Shape: (1338401, 37)


In [4]:
print(df_title_body["repository_id"].nunique())

59474


## 3. Define an initial PR taxonomy

These keyword groups are heuristic labels, not final ground truth. They provide a fast baseline, an interpretable comparison point for discovered topics, and a way to identify examples for manual review.

In [5]:
PR_CATEGORIES = {
    "feature": [
        "add", "implement", "introduce", "support", "enable",
        "new feature", "create",
    ],
    "bug_fix": [
        "fix", "bug", "resolve", "correct", "patch",
        "hotfix", "regression",
    ],
    "refactor": [
        "refactor", "cleanup", "clean up", "simplify",
        "restructure", "reorganize",
    ],
    "documentation": [
        "docs", "documentation", "readme", "comment",
        "guide", "tutorial",
    ],
    "testing": [
        "test", "tests", "testing", "coverage",
        "unit test", "integration test",
    ],
    "dependency": [
        "dependency", "dependencies", "bump", "upgrade",
        "downgrade", "package", "version",
    ],
    "security": [
        "security", "vulnerability", "cve", "authentication",
        "authorization", "permission", "sanitize",
    ],
    "performance": [
        "performance", "optimize", "optimization", "speed",
        "latency", "memory", "cache",
    ],
    "configuration": [
        "config", "configuration", "settings",
        "environment variable", "yaml",
    ],
    "release": [
        "release", "changelog", "publish", "tag",
        "prepare release",
    ],
    "revert": [
        "revert", "rollback", "back out", "undo",
    ],
}

PR_CATEGORIES

{'feature': ['add',
  'implement',
  'introduce',
  'support',
  'enable',
  'new feature',
  'create'],
 'bug_fix': ['fix',
  'bug',
  'resolve',
  'correct',
  'patch',
  'hotfix',
  'regression'],
 'refactor': ['refactor',
  'cleanup',
  'clean up',
  'simplify',
  'restructure',
  'reorganize'],
 'documentation': ['docs',
  'documentation',
  'readme',
  'comment',
  'guide',
  'tutorial'],
 'testing': ['test',
  'tests',
  'testing',
  'coverage',
  'unit test',
  'integration test'],
 'dependency': ['dependency',
  'dependencies',
  'bump',
  'upgrade',
  'downgrade',
  'package',
  'version'],
 'security': ['security',
  'vulnerability',
  'cve',
  'authentication',
  'authorization',
  'permission',
  'sanitize'],
 'performance': ['performance',
  'optimize',
  'optimization',
  'speed',
  'latency',
  'memory',
  'cache'],
 'configuration': ['config',
  'configuration',
  'settings',
  'environment variable',
  'yaml'],
 'release': ['release', 'changelog', 'publish', 'tag', 'p

## 4. Match PRs by taxonomy keywords

Apply the initial taxonomy to the pull request text and assign one or more keyword-based categories to each PR.

In [ ]:
from tqdm import tqdm

tqdm.pandas()


def keyword_pattern(keywords: list[str]) -> re.Pattern:

    escaped_keywords = [
        re.escape(keyword.lower())
        for keyword in keywords
    ]

    keyword_regex = r"|".join(
        rf"(?<!\w){keyword.replace(r'\ ', r'\s+')}(?!\w)"
        for keyword in escaped_keywords
    )

    return re.compile(
        keyword_regex,
        flags=re.IGNORECASE
    )


# ============================================================
# BUILD CATEGORY REGEX PATTERNS
# ============================================================

category_patterns = {
    category: keyword_pattern(keywords)
    for category, keywords in PR_CATEGORIES.items()
}


# ============================================================
# FIND TEXT COLUMNS
# ============================================================

text_columns = [
    column
    for column in [
        "title",
        "body",
        "pr_title",
        "pr_body",
        "clean_title",
        "clean_body"
    ]
    if column in df_title_body.columns
]

if not text_columns:
    raise ValueError(
        "No title/body text columns found for keyword matching."
    )


# ============================================================
# COMBINE PR TEXT
# ============================================================

print("Combining PR text...")

pr_text = (
    df_title_body[text_columns]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.lower()
)


# ============================================================
# KEYWORD MATCHING WITH PROGRESS BAR
# ============================================================

print("\nMatching keyword categories...")

for category, pattern in tqdm(
    category_patterns.items(),
    total=len(category_patterns),
    desc="Categories"
):

    df_title_body[f"kw_{category}"] = (
        pr_text.str.contains(
            pattern,
            regex=True
        )
    )


# ============================================================
# MATCHED CATEGORY LIST
# ============================================================

def matched_categories(row: pd.Series) -> list[str]:

    return [
        category
        for category in PR_CATEGORIES
        if row[f"kw_{category}"]
    ]


keyword_score_columns = [
    f"kw_{category}"
    for category in PR_CATEGORIES
]


print("\nBuilding category lists for each PR...")

df_title_body["keyword_categories"] = (
    df_title_body[keyword_score_columns]
    .progress_apply(
        matched_categories,
        axis=1
    )
)


# ============================================================
# CATEGORY COUNTS
# ============================================================

df_title_body["keyword_category_count"] = (
    df_title_body[keyword_score_columns]
    .sum(axis=1)
)

df_title_body["has_keyword_match"] = (
    df_title_body["keyword_category_count"]
    .gt(0)
)


# ============================================================
# CATEGORY DISTRIBUTION
# ============================================================

keyword_distribution = (
    df_title_body[keyword_score_columns]
    .sum()
    .rename_axis("category")
    .reset_index(name="matched_prs")
)

keyword_distribution["category"] = (
    keyword_distribution["category"]
    .str.removeprefix("kw_")
)

keyword_distribution["share_of_prs"] = (
    keyword_distribution["matched_prs"]
    / len(df_title_body)
)

keyword_distribution = (
    keyword_distribution
    .sort_values(
        "matched_prs",
        ascending=False
    )
)


# ============================================================
# RESULTS
# ============================================================

print("\nDone.")

print(
    "Text columns used:",
    text_columns
)

print(
    "PRs with at least one keyword match:",
    df_title_body["has_keyword_match"].sum()
)

keyword_distribution

Combining PR text...

Matching keyword categories...


Categories:  18%|█▊        | 2/11 [06:31<28:37, 190.85s/it]

In [ ]:
# ============================================================
# VERIFY PR CATEGORIES
# ============================================================

category_cols = [
    col for col in df_title_body.columns
    if col.startswith("kw_")
]

print("CATEGORY CHECK")
print("=" * 50)

print(f"Total PRs: {len(df_title_body):,}")
print(f"Number of category columns: {len(category_cols):,}")

print("\nCategory columns:")
print(category_cols)


# Make sure expected columns exist
required_category_cols = [
    "keyword_categories",
    "keyword_category_count",
    "has_keyword_match"
]

for col in required_category_cols:
    print(f"{col}: {'FOUND' if col in df_title_body.columns else 'MISSING'}")


# ------------------------------------------------------------
# How many PRs have categories?
# ------------------------------------------------------------

matched = df_title_body["has_keyword_match"].sum()
unmatched = (~df_title_body["has_keyword_match"]).sum()

print("\nCATEGORY COVERAGE")
print("=" * 50)

print(f"PRs with at least one category: {matched:,}")
print(f"PRs with no category:           {unmatched:,}")
print(f"Coverage:                       {matched / len(df_title_body):.2%}")


# ------------------------------------------------------------
# Category distribution
# ------------------------------------------------------------

category_distribution = (
    df_title_body[category_cols]
    .sum()
    .sort_values(ascending=False)
    .rename("PR_count")
    .to_frame()
)

category_distribution["percent_of_PRs"] = (
    category_distribution["PR_count"] / len(df_title_body) * 100
)

display(category_distribution)


# ------------------------------------------------------------
# Distribution of number of categories per PR
# ------------------------------------------------------------

print("\nNUMBER OF CATEGORIES PER PR")
display(
    df_title_body["keyword_category_count"]
    .value_counts()
    .sort_index()
    .rename("PR_count")
    .to_frame()
)


# ------------------------------------------------------------
# Look at actual examples
# ------------------------------------------------------------

columns_to_show = [
    col for col in [
        "pull_request_id",
        "title",
        "keyword_categories",
        "keyword_category_count"
    ]
    if col in df_title_body.columns
]

display(
    df_title_body.loc[
        df_title_body["has_keyword_match"],
        columns_to_show
    ].sample(
        min(20, matched),
        random_state=42
    )
)


# ------------------------------------------------------------
# Sanity check:
# keyword_category_count should equal sum of kw_* columns
# ------------------------------------------------------------

assert (
    df_title_body["keyword_category_count"]
    == df_title_body[category_cols].sum(axis=1)
).all(), "Category counts do not match category columns!"

print("\n✓ Category validation passed.")


# ------------------------------------------------------------
# SAVE FULL DATAFRAME INCLUDING CATEGORIES
# ------------------------------------------------------------

output_file = "../data/microsoft/df_title_body_with_categories.csv" 

df_title_body.to_csv(
    output_file,
    index=False
)

print(f"\nSaved {len(df_title_body):,} PRs")
print(f"Saved {len(df_title_body.columns):,} columns")
print(f"File: {output_file}")

In [3]:
df_title_body = pd.read_csv("../data/df_title_body.csv")

print(df_title_body.shape)
df_title_body.head()

print("Rows:", df_title_body.shape[0])
print("Columns:", df_title_body.shape[1])
print("Shape:", df_title_body.shape)

(1338401, 51)
Rows: 1338401
Columns: 51
Shape: (1338401, 51)


## 5. Calculate keyword-match coverage

Compute the share of pull requests that received at least one keyword-based category.

In [4]:
# ------------------------------------------------------------
# How many PRs have categories?
# ------------------------------------------------------------

matched = df_title_body["has_keyword_match"].sum()
unmatched = (~df_title_body["has_keyword_match"]).sum()

print("\nCATEGORY COVERAGE")
print("=" * 50)

print(f"PRs with at least one category: {matched:,}")
print(f"PRs with no category:           {unmatched:,}")
print(f"Coverage:                       {matched / len(df_title_body):.2%}")


CATEGORY COVERAGE
PRs with at least one category: 1,231,185
PRs with no category:           107,216
Coverage:                       91.99%


## 6. Compare PR categories with time to merge

Summarize how long pull requests take to merge across keyword-based categories. Because one PR can match multiple categories, this analysis counts a PR once per matched category.

In [ ]:
from pathlib import Path
import pandas as pd

data_dir = Path("../data/microsoft")
file_path = data_dir / "df_title_body_with_categories.csv"

df_title_body = pd.read_csv(file_path)

print(f"Dataframe shape: {df_title_body.shape}")
print("\nColumns:")
print(df_title_body.columns.tolist())

df_title_body.head()

In [ ]:
# Get all keyword category columns
keyword_cols = [
    col for col in df_title_body.columns
    if col.startswith("kw_")
]

# Count how many PRs matched each category
category_counts = (
    df_title_body[keyword_cols]
    .sum()
    .rename(lambda x: x.removeprefix("kw_"))
    .sort_values()
)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

ax.barh(
    category_counts.index,
    category_counts.values,
    color="#0ea31b",
    edgecolor="black"
)

ax.set_title("Number of PRs by Category")
ax.set_xlabel("Number of PRs")
ax.set_ylabel("PR Category")

ax.grid(axis="x", alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PR CATEGORY COUNTS - SEPARATE PLOT FOR EACH MONTH
# ============================================================

keyword_cols = [
    col for col in df_title_body.columns
    if col.startswith("kw_")
]

# Pick your date column
date_col = "merge_day"

# Convert to datetime
df_title_body[date_col] = pd.to_datetime(
    df_title_body[date_col],
    errors="coerce"
)

# Create month column
df_title_body["month"] = (
    df_title_body[date_col]
    .dt.to_period("M")
    .astype(str)
)

# Create a separate plot for each month
for month in sorted(df_title_body["month"].dropna().unique()):

    month_df = df_title_body[
        df_title_body["month"] == month
    ]

    category_counts = (
        month_df[keyword_cols]
        .sum()
        .rename(lambda x: x.removeprefix("kw_"))
        .sort_values()
    )

    fig, ax = plt.subplots(figsize=(10, 6))

    ax.barh(
        category_counts.index,
        category_counts.values,
        color="#0ea31b",
        edgecolor="black"
    )

    ax.set_title(f"Number of PRs by Category - {month}")
    ax.set_xlabel("Number of PRs")
    ax.set_ylabel("PR Category")

    ax.grid(axis="x", alpha=0.25)

    plt.tight_layout()
    plt.show()

In [ ]:
merge_time_column = "minutes_to_merge"

if merge_time_column not in df_title_body.columns:
    raise ValueError(f"Expected '{merge_time_column}' in df_title_body.")

merge_time_df = df_title_body.copy()
merge_time_df[merge_time_column] = pd.to_numeric(
    merge_time_df[merge_time_column],
    errors="coerce",
)
merge_time_df = merge_time_df[merge_time_df[merge_time_column].notna()]
merge_time_df = merge_time_df[merge_time_df[merge_time_column].ge(0)]
merge_time_df["hours_to_merge"] = merge_time_df[merge_time_column] / 60
merge_time_df["days_to_merge"] = merge_time_df["hours_to_merge"] / 24

category_merge_time = (
    merge_time_df.loc[merge_time_df["has_keyword_match"], [
        "pull_request_id",
        "keyword_categories",
        "hours_to_merge",
        "days_to_merge",
    ]]
    .explode("keyword_categories")
    .rename(columns={"keyword_categories": "category"})
)

category_merge_summary = (
    category_merge_time
    .groupby("category")
    .agg(
        matched_prs=("pull_request_id", "nunique"),
        median_hours_to_merge=("hours_to_merge", "median"),
        mean_hours_to_merge=("hours_to_merge", "mean"),
        p75_hours_to_merge=("hours_to_merge", lambda values: values.quantile(0.75)),
        median_days_to_merge=("days_to_merge", "median"),
    )
    .reset_index()
    .sort_values("median_hours_to_merge")
)

overall_merge_summary = pd.DataFrame({
    "metric": [
        "PRs with merge time",
        "Median hours to merge",
        "Mean hours to merge",
        "Median days to merge",
    ],
    "value": [
        len(merge_time_df),
        merge_time_df["hours_to_merge"].median(),
        merge_time_df["hours_to_merge"].mean(),
        merge_time_df["days_to_merge"].median(),
    ],
})

display(overall_merge_summary)
display(category_merge_summary)

import ast
import matplotlib.pyplot as plt

category_df = df_title_body[
    ["keyword_categories", "minutes_to_merge"]
].dropna().copy()

def parse_categories(x):
    if isinstance(x, list):
        return x

    if isinstance(x, str):
        x = x.strip()

        if x.startswith("["):
            try:
                return ast.literal_eval(x)
            except (ValueError, SyntaxError):
                return []

        return [x]

    return []

category_df["keyword_categories"] = (
    category_df["keyword_categories"].apply(parse_categories)
)

category_df = category_df.explode(
    "keyword_categories",
    ignore_index=True
)

category_merge_summary = (
    category_df
    .groupby("keyword_categories", as_index=False)["minutes_to_merge"]
    .median()
)

category_merge_summary["median_hours_to_merge"] = (
    category_merge_summary["minutes_to_merge"] / 60
)

plot_data = category_merge_summary.sort_values(
    "median_hours_to_merge",
    ascending=True
)

fig, ax = plt.subplots(figsize=(7, 4))

ax.barh(
    plot_data["keyword_categories"],
    plot_data["median_hours_to_merge"],
    color="#0ea31b",
    edgecolor="black"
)

ax.set_title("Median Time to Merge by PR Category")
ax.set_xlabel("Median Hours to Merge")
ax.set_ylabel("PR Category")

ax.grid(axis="x", alpha=0.25)
ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

## 7. Check whether category merge-time patterns hold by primary language

Repeat the category-versus-time-to-merge view for large primary programming languages. The summary table compares each language-specific category pattern with the overall pattern so it is easier to see where the relationship is stable versus language-specific.

In [ ]:
language_candidates = ["JavaScript", "Python", "Java", "C++"]
language_top_n_categories = 12
minimum_language_category_prs = 30

language_source = df_title_body.copy()
required_columns = {"pull_request_id", "languages_list", "minutes_to_merge", "keyword_categories", "has_keyword_match"}
missing_columns = required_columns.difference(language_source.columns)
if missing_columns:
    raise ValueError(f"Missing columns required for language-filtered analysis: {sorted(missing_columns)}")


def parse_language_list(value: object) -> list[str]:
    if isinstance(value, float) and pd.isna(value):
        return []
    if isinstance(value, (list, tuple, set, np.ndarray)):
        return [str(language).strip() for language in value if str(language).strip()]

    text_value = str(value).strip()
    if not text_value or text_value == "[]":
        return []

    quoted_languages = re.findall(r"'([^']+)'|\"([^\"]+)\"", text_value)
    if quoted_languages:
        return [
            (single_quoted or double_quoted).strip()
            for single_quoted, double_quoted in quoted_languages
            if (single_quoted or double_quoted).strip()
        ]

    return [
        language.strip().strip("'\"")
        for language in text_value.strip("[]").split(",")
        if language.strip().strip("'\"")
    ]


language_source["programming_languages"] = language_source["languages_list"].apply(parse_language_list)
language_source["primary_programming_language"] = language_source["programming_languages"].apply(
    lambda languages: languages[0] if languages else np.nan
)
language_source["hours_to_merge_for_language"] = pd.to_numeric(
    language_source["minutes_to_merge"],
    errors="coerce",
) / 60
language_source = language_source[
    language_source["hours_to_merge_for_language"].notna()
    & language_source["hours_to_merge_for_language"].ge(0)
    & language_source["has_keyword_match"]
]

language_category_time = (
    language_source[
        [
            "pull_request_id",
            "primary_programming_language",
            "keyword_categories",
            "hours_to_merge_for_language",
        ]
    ]
    .explode("keyword_categories")
    .rename(columns={"keyword_categories": "category"})
)
language_category_time = language_category_time[
    language_category_time["primary_programming_language"].isin(language_candidates)
    & language_category_time["category"].notna()
]

overall_category_medians = (
    category_merge_summary
    .set_index("category")["median_hours_to_merge"]
    .rename("overall_median_hours_to_merge")
)

language_category_summary = (
    language_category_time
    .groupby(["primary_programming_language", "category"])
    .agg(
        matched_prs=("pull_request_id", "nunique"),
        median_hours_to_merge=("hours_to_merge_for_language", "median"),
        mean_hours_to_merge=("hours_to_merge_for_language", "mean"),
        p75_hours_to_merge=("hours_to_merge_for_language", lambda values: values.quantile(0.75)),
    )
    .reset_index()
)
language_category_summary = language_category_summary[
    language_category_summary["matched_prs"].ge(minimum_language_category_prs)
].copy()
language_category_summary = language_category_summary.merge(
    overall_category_medians,
    on="category",
    how="left",
)

language_pattern_rows = []
for language in language_candidates:
    language_rows = language_category_summary[
        language_category_summary["primary_programming_language"].eq(language)
    ].copy()
    compared_rows = language_rows.dropna(subset=["overall_median_hours_to_merge"])
    rank_correlation = compared_rows[[
        "median_hours_to_merge",
        "overall_median_hours_to_merge",
    ]].corr(method="spearman").iloc[0, 1] if len(compared_rows) >= 3 else np.nan

    if language_rows.empty:
        language_pattern_rows.append({
            "language": language,
            "primary_language_prs": int(
                language_source["primary_programming_language"].eq(language).sum()
            ),
            "categories_compared": 0,
            "spearman_r_vs_overall": np.nan,
            "fastest_category": np.nan,
            "slowest_category": np.nan,
            "median_hour_range": np.nan,
        })
        continue

    fastest_row = language_rows.loc[language_rows["median_hours_to_merge"].idxmin()]
    slowest_row = language_rows.loc[language_rows["median_hours_to_merge"].idxmax()]
    language_pattern_rows.append({
        "language": language,
        "primary_language_prs": int(
            language_source["primary_programming_language"].eq(language).sum()
        ),
        "categories_compared": len(compared_rows),
        "spearman_r_vs_overall": rank_correlation,
        "fastest_category": fastest_row["category"],
        "slowest_category": slowest_row["category"],
        "median_hour_range": slowest_row["median_hours_to_merge"] - fastest_row["median_hours_to_merge"],
    })

language_pattern_summary = pd.DataFrame(language_pattern_rows)
language_pattern_display = language_pattern_summary.assign(
    spearman_r_vs_overall=lambda frame: frame["spearman_r_vs_overall"].round(2),
    median_hour_range=lambda frame: frame["median_hour_range"].round(1),
)
display(language_pattern_display)

display(
    language_category_summary
    .sort_values(["primary_programming_language", "median_hours_to_merge"])
    .reset_index(drop=True)
)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for ax, language in zip(axes, language_candidates):
    plot_data = (
        language_category_summary[
            language_category_summary["primary_programming_language"].eq(language)
        ]
        .sort_values("median_hours_to_merge", ascending=True)
        .tail(language_top_n_categories)
        .sort_values("median_hours_to_merge", ascending=True)
    )

    if plot_data.empty:
        ax.text(0.5, 0.5, f"No categories with >= {minimum_language_category_prs} PRs", ha="center", va="center")
        ax.set_title(language)
        ax.axis("off")
        continue

    language_stats = language_pattern_summary.set_index("language").loc[language]
    plot_limit = plot_data["median_hours_to_merge"].max() * 1.18
    ax.barh(plot_data["category"], plot_data["median_hours_to_merge"], color="#3a7ca5")
    for row_index, row in enumerate(plot_data.itertuples(index=False)):
        ax.text(
            row.median_hours_to_merge + plot_limit * 0.01,
            row_index,
            f"{row.matched_prs:,}",
            va="center",
            fontsize=9,
        )
    ax.set_title(
        f"{language}: median merge time by PR category\n"
        f"Spearman r vs overall = {language_stats.spearman_r_vs_overall:.2f}"
    )
    ax.set_xlabel("Median hours to merge")
    ax.grid(axis="x", alpha=0.25)
    ax.set_xlim(0, plot_limit)

for ax in axes[len(language_candidates):]:
    ax.axis("off")

fig.suptitle("Does the category merge-time pattern hold within primary programming languages?", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Plot 7: Merge Time by Language - Clear Bar Chart

import seaborn as sns
import ast

if "primary_language" not in df_title_body.columns and "languages_list" in df_title_body.columns:
    def parse_languages(value):
        if isinstance(value, list):
            return value
        if isinstance(value, str):
            try:
                parsed = ast.literal_eval(value)
            except (ValueError, SyntaxError):
                return []
            return parsed if isinstance(parsed, list) else []
        return []

    df_title_body["languages_list"] = df_title_body["languages_list"].apply(parse_languages)
    df_title_body["primary_language"] = df_title_body["languages_list"].apply(
        lambda x: x[0] if len(x) > 0 else None
    )

if "primary_language" not in df_title_body.columns:
    print("No primary_language column could be derived. Available columns:", df_title_body.columns.tolist())
elif "minutes_to_merge" not in df_title_body.columns:
    print("No minutes_to_merge column found. Available columns:", df_title_body.columns.tolist())
else:
    top_langs_count = df_title_body["primary_language"].dropna().value_counts().head(12).index
    merge_time_data = df_title_body[df_title_body["primary_language"].isin(top_langs_count)].copy()
    merge_time_data["minutes_to_merge_hours"] = pd.to_numeric(
        merge_time_data["minutes_to_merge"],
        errors="coerce"
    ) / 60
    merge_time_data = merge_time_data.dropna(subset=["minutes_to_merge_hours"])

    lang_median_merge = (
        merge_time_data
        .groupby("primary_language")["minutes_to_merge_hours"]
        .agg(["median", "mean", "count"])
        .sort_values("median", ascending=False)
    )

    fig, ax = plt.subplots(figsize=(10, 5))

    x_pos = range(len(lang_median_merge))
    colors_gradient = sns.color_palette("RdYlGn_r", len(lang_median_merge))[::-1]
    bars = ax.barh(
        x_pos,
        lang_median_merge["median"],
        color=sns.color_palette("Greens", len(lang_median_merge)),
        edgecolor="black",
        linewidth=1.2,
        alpha=0.8
    )

    ax.set_yticks(x_pos)
    ax.set_yticklabels(lang_median_merge.index, fontsize=11)
    ax.set_xlabel("Median Time to Merge (hours)", fontsize=12, fontweight="bold")
    ax.set_title("How Fast Do PRs Merge by Language? (Median Time)", fontsize=14, fontweight="bold", pad=20)
        
    ax.tick_params(axis="x", labelsize=10)
    ax.grid(axis="x", alpha=0.3)
    ax.set_axisbelow(True)
    ax.set_xlim(0, lang_median_merge["median"].max() * 1.15)

    plt.tight_layout()
    plt.show()